In [1]:
# Three different sales agents for three different types of cold emails - Professional, Engaging, Busy
# Convert them into tools as sales_agents
# Create an agent for email subject writer and another agent to convert from text to html
# Write a function for send_email and convert it into tool
# Create an agent for email_manager that uses tools - subject_writer, html_converter, semd_email
# Create another agent for sales_manager and pass the email_manager as handoff

In [2]:
import os
import asyncio

from typing import Dict

from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, trace

import sendgrid
from sendgrid.helpers.mail import Mail, Content, To, Email

In [3]:
load_dotenv(override=True)

True

In [4]:
# Instructions for sales agents

instruction1 = "You are a sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium business. \
You write professional, serious cold emails"

instruction2 = "You are a humorous, engaging sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write witty, engaging cold emails that are likely to get a response."

instruction3 = "You are a busy sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write concise, to the point cold emails"


In [5]:
# Creating sales agents

sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instruction1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instruction2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instruction1,
    model="gpt-4o-mini"
)


In [6]:
# Sales Agents are being converted in 'tools'
description = "Write a cold sales email"

sa1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
sa2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
sa3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [7]:
# Sales agent tools are put into a simple list
sales_agents = [sa1, sa2, sa3]

In [8]:
# Creating a tool to write email subjects

subject_instructions = "You can write a subject for a cold sales email. You are given a message \
and you need to write a subject for an email that is most likely to get a response"

subject_writer = Agent(
    name="Email Subject Writer",
    instructions=subject_instructions,
    model="gpt-4o-mini"
)

subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

In [9]:
# Creating a tool to convert text into html email body

html_instructions = "You can convert an email body from text to html. You are given a cold sales \
body that may contain markdown. You need to convert the text email into html email with simple, clear, \
compelling layout and design"

html_converter = Agent(
    name="HTML email body converter",
    instructions=html_instructions,
    model="gpt-4o-mini"
)

html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body into HTML email body")

In [10]:
# send email tool
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    from_email = Email("wetechfin@gmail.com")
    to_email = To("debmalyamondal63@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email=from_email, to_emails=to_email, subject=subject, html_content=content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status", "success"}

In [11]:
formatter_tool = [subject_tool, html_tool, send_html_email]

In [12]:
# Creating agent for email manager

em_instructions = "You are a sales email manager and sender. You receive the body of an email to be \
sent. You first use the subject_tool to write the subject of the email, then use the html_tool to convert \
the body to HTML. Finally, you use the send_html_email tool to send the email with the html body and the\
subject"

emailer_agent = Agent(
    name="Sales_Email_Manager",
    instructions=em_instructions,
    model="gpt-4o-mini",
    tools=formatter_tool,
    handoff_description="Convert an email to HTML and send it"
)

In [13]:
handoffs = [emailer_agent]

In [14]:
instructions = """
You are a sales manager at the Automation Agency company AutAI. Your goal is to find the single best
sales email using the sales_agent tools.
Follow these instructions carefully:
1. Generate Drafts: Use all three sales_agents tools to generate three different cold email drafts. 
Do not proceed until all three drafts are ready.
2. Evaluate and Select: Review and select the best email using your judgement of which one is more effective.
3. Handoff for sending: Pass only the winning email to the handoff agent. The handoff agent will take care of
formatting and sending the email

Crucial Rules:
- You must use the sales_agent tools to generate the drafts. Do not write them yourself.
- You must send one email using send_email tool and not more than one.

"""

In [15]:
sales_manager = Agent(
    name="sales_manager",
    instructions=instructions,
    tools=sales_agents,
    model="gpt-4o-mini",
    handoffs=handoffs
)

In [16]:
message = "Send out a cold email with Dear CEO from Debmalya"

with trace("Automated AutAI Sales Campaign"):
    result = await Runner.run(sales_manager, message)

In [17]:
result.final_output

'The cold email has been successfully sent! If you need any further assistance or additional emails, just let me know!'